# Getting Started

This notebook gets you running the active learning pipeline with the ESC10 demo dataset. By the end you will have:

1. Run a set of predefined experiments from the config file
2. Compared the results of different sampling strategies
3. Understood how to write your own custom sampling strategy

If you are competing in the BioDCASE challenge, this is a good starting point. You can swap the ESC10 dataset for your own data once you understand the pipeline.

## Prerequisites

Make sure you have installed the project dependencies:

```bash
uv sync
```

And that you have the `ESC10_BASEAL/` dataset folder in the project root with the following structure:

```
ESC10_BASEAL/
├── data/
│   └── birdnet/
│       └── *.mp3                    # audio segments (Only used in app)
├── embeddings/
│   └── birdnet/
│       └── {filename}_birdnet.npy   # pre-computed embeddings
├── labels.csv                       # filename, label, validation
└── metadata.csv                     # detailed segment metadata
```

> **Note:** Everything related to active learning is within `core/`. The `app/` and `api/` directories can be set up separately if you wish to use the web interface.

## 1. Running experiments from the config

The simplest way to run experiments is through the `Manager` class, which reads experiment definitions from `core/config.yml`.

Open `core/config.yml` and you will see something like this:

```yaml
experiments:
  - name: Random
    embeddings_dir: ESC10_BASEAL/embeddings/birdnet/
    annotations_path: ESC10_BASEAL/labels.csv
    hidden_dim: null
    sampling_strategy: random
    learning_rate: 0.001
    repeats: 2
    device: cpu

  - name: Margin
    embeddings_dir: ESC10_BASEAL/embeddings/birdnet/
    annotations_path: ESC10_BASEAL/labels.csv
    hidden_dim: null
    sampling_strategy: margin
    learning_rate: 0.001
    repeats: 2
    device: cpu
```

Each entry defines an experiment with a sampling strategy. The `Manager` loads all of these and runs them in sequence (or parallel).

In [ ]:
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path().absolute().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from core.active_learner import Manager

# Load experiments from config
config_path = PROJECT_ROOT / "core" / "config.yml"
manager = Manager(config_path, base_dir=PROJECT_ROOT, verbose=False)

print(f"Loaded {len(manager.experiments)} experiments:")
for name in manager.experiment_names:
    print(f"  - {name}")

### Run active learning cycles

Each cycle does three things: sample new points, add them to the labelled set, and train the model. We run several cycles to see how performance improves.

In [ ]:
n_cycles = 10

for cycle in range(n_cycles):
    results = manager.run(
        n_samples=10,
        epochs=10,
        batch_size=8,
        parallel=False,
    )

    if cycle % 3 == 0 or cycle == n_cycles - 1:
        print(f"\nCycle {cycle + 1}/{n_cycles}")
        for exp_name, metrics in results.items():
            print(f"  {exp_name}: accuracy={metrics['accuracy']:.4f}, loss={metrics['loss']:.4f}, labelled={metrics['n_labeled']}")

## 2. Comparing results

Plot the accuracy curves to see which sampling strategy performs best with fewer labels.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for learner, name in zip(manager.experiments, manager.experiment_names):
    accuracies = [h['accuracy'] for h in learner.training_history]
    losses = [h['loss'] for h in learner.training_history]
    n_labeled = [h['n_labeled'] for h in learner.training_history]

    axes[0].plot(n_labeled, accuracies, marker='o', label=name)
    axes[1].plot(n_labeled, losses, marker='o', label=name)

axes[0].set_xlabel('Labelled samples')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs labelled samples')
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel('Labelled samples')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss vs labelled samples')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Save results

Results are saved as JSON files with training history and experiment metadata.

In [ ]:
manager.save(output_dir=PROJECT_ROOT / "results" / "getting_started")
print("Results saved.")

## 3. Writing a custom sampling strategy

The predefined strategies (random, margin, coreset, nn_disagreement) are a good starting point, but you may want to try something different.

Open `core/utils/sampling.py` and find the `_custom()` method inside the `SamplingStrategy` class. This is where you write your own logic.

The method must return a numpy array of **utility scores** for each unlabelled sample. Higher scores mean the sample is more useful to label. The framework will pick the top-scoring samples for you.

### What you have access to:

Inside `_custom()`, you can use these instance attributes (set by the `select()` method each cycle):

| Attribute | Type | Description |
|-----------|------|-------------|
| `self.unlabeled_indices` | list of int | Indices of the unlabelled pool |
| `self.predictions` | ndarray (N, num_classes) | Model probability predictions for all samples |
| `self.embeddings` | ndarray (N, embedding_dim) | Feature embeddings for all samples |
| `self.labeled_indices` | list of int | Indices of already labelled samples |
| `self.labels` | ndarray | Ground-truth labels for all samples |
| `self.model` | nn.Module | The trained model (if you need gradients or features) |
| `self.annotations` | DataFrame | Annotation metadata |

### Example: entropy sampling

Here is what a simple entropy-based strategy would look like:

```python
def _custom(self) -> np.ndarray:
    if self.predictions is None:
        return self._random()

    unlabeled_preds = self.predictions[self.unlabeled_indices]

    # Compute entropy: higher entropy = more uncertain
    epsilon = 1e-10
    entropy = -np.sum(unlabeled_preds * np.log(unlabeled_preds + epsilon), axis=1)

    # Normalise to [0, 1]
    utility = self._normalize(entropy)
    return utility
```

### Using your custom strategy

Once you have written your `_custom()` method, set `sampling_strategy: custom` in `core/config.yml`:

```yaml
experiments:
  - name: MyCustomStrategy
    embeddings_dir: ESC10_BASEAL/embeddings/birdnet/
    annotations_path: ESC10_BASEAL/labels.csv
    hidden_dim: null
    sampling_strategy: custom
    learning_rate: 0.001
    repeats: 2
    device: cpu
```

Then run the pipeline exactly as above. Your strategy will be compared against the others.

### Tips for writing strategies

- Return values should be in the range [0, 1]. Use `self._normalize(scores)` to handle this.
- The framework selects the top-k samples by utility score, so you do not need to handle selection logic yourself.
- You can combine multiple signals (e.g. uncertainty + diversity) by averaging or weighting normalised scores.
- Look at `_margin()` and `_coreset_farthest()` in `sampling.py` for reference implementations.

## Next steps

- **Understand the data format** -- see notebook 02 for how embeddings and labels are loaded.
- **Understand the model** -- see notebook 03 for how the classifier works.
- **Understand the AL loop** -- see notebook 04 for the step-by-step active learning cycle.
- **Full pipeline walkthrough** -- see notebook 05 for initialising and running a single experiment.
- **Experiment manager** -- see notebook 06 for running and comparing multiple experiments.
- **Use your own data** -- replace the `ESC10_BASEAL/` folder with your own embeddings and labels CSV.